In [2]:
# COMMAND: Install all required packages

!pip install -q pandas numpy scikit-learn joblib

print("Required packages installed successfully!")

Required packages installed successfully!


In [4]:
# COMMAND: Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [5]:
# COMMAND: Import required Python libraries

import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [6]:
# COMMAND: Define Google Drive project paths

import os

project_path = "/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster"

dataset_path = os.path.join(
    project_path,
    "data",
    "alumni_donor_dataset_2025.csv"
)

model_path = os.path.join(
    project_path,
    "models",
    "alumni_donor_model_pipeline.pkl"
)

results_path = os.path.join(
    project_path,
    "results",
    "model_training_results.csv"
)

print("Project Path:")
print(project_path)

print("\nDataset Path:")
print(dataset_path)

print("\nModel Path:")
print(model_path)

print("\nResults Path:")
print(results_path)

Project Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster

Dataset Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/data/alumni_donor_dataset_2025.csv

Model Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl

Results Path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/results/model_training_results.csv


In [7]:
# COMMAND: Create required project folders

import os

os.makedirs(
    os.path.join(project_path, "data"),
    exist_ok=True
)

os.makedirs(
    os.path.join(project_path, "models"),
    exist_ok=True
)

os.makedirs(
    os.path.join(project_path, "results"),
    exist_ok=True
)

print("All project folders are ready!")

All project folders are ready!


In [8]:
# COMMAND: Verify that the dataset exists

import os

if os.path.exists(dataset_path):

    print("SUCCESS!")
    print("Dataset found at:")
    print(dataset_path)

else:

    print("ERROR!")
    print("Dataset was not found.")
    print("Expected location:")
    print(dataset_path)

SUCCESS!
Dataset found at:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/data/alumni_donor_dataset_2025.csv


In [9]:
# COMMAND: Load the alumni dataset

import pandas as pd

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")

print("\nDataset Shape:")
print(df.shape)

print("\nNumber of Rows:", len(df))

print("Number of Columns:", len(df.columns))

Dataset loaded successfully!

Dataset Shape:
(10000, 18)

Number of Rows: 10000
Number of Columns: 18


In [10]:
# COMMAND: Display first five records

import pandas as pd

print("First 5 Alumni Records:")

display(df.head())

First 5 Alumni Records:


,Alumni_ID,Age,Gender,Graduation_Year,Degree_Level,Major,Alumni_Status,Email_Available,Phone_Available,Wealth_Rating,Income_Bracket,Event_Attendance,Email_Open_Rate,Location,Consecutive_Giving_Years,Donation_Last_Year_2024,Total_Lifetime_Giving,Is_Donor_2025
0,ALM000001,52,Male,1973,Bachelor,Education,Inactive,Yes,Yes,C,<$50K,6,42.75,"Houston, USA",13,66.16,1025.70,Yes
1,ALM000002,42,Male,2023,Master,Business,Inactive,Yes,Yes,B,$100K-$150K,0,33.97,"Nottingham, UK",4,1326.32,6355.61,No
2,ALM000003,54,Male,1973,Bachelor,Business,Active,Yes,Yes,C,$50K-$100K,0,28.91,"Portland, USA",17,57.23,1106.37,Yes
3,ALM000004,67,Male,1997,Master,Law,Lost Contact,Yes,Yes,B+,>$500K,4,40.08,"Quebec City, Canada",0,0.00,0.00,Yes
4,ALM000005,41,Female,1972,Bachelor,Liberal Arts,Inactive,Yes,Yes,B+,$50K-$100K,4,11.41,"Shenzhen, China",0,0.00,0.00,No


In [11]:
# COMMAND: Display dataset information

print("Dataset Information:")
print("=" * 60)

df.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Alumni_ID                 10000 non-null  object 
 1   Age                       10000 non-null  int64  
 2   Gender                    10000 non-null  object 
 3   Graduation_Year           10000 non-null  int64  
 4   Degree_Level              10000 non-null  object 
 5   Major                     10000 non-null  object 
 6   Alumni_Status             10000 non-null  object 
 7   Email_Available           10000 non-null  object 
 8   Phone_Available           10000 non-null  object 
 9   Wealth_Rating             10000 non-null  object 
 10  Income_Bracket            10000 non-null  object 
 11  Event_Attendance          10000 non-null  int64  
 12  Email_Open_Rate           10000 non-null  float64
 13  Location                  10000 non-null 

In [12]:

# COMMAND: Check missing values in every column

missing_values = df.isnull().sum()

print("Missing Values:")
print("=" * 60)

print(missing_values)

print("\nTotal Missing Values:")
print(missing_values.sum())

Missing Values:
Alumni_ID                   0
Age                         0
Gender                      0
Graduation_Year             0
Degree_Level                0
Major                       0
Alumni_Status               0
Email_Available             0
Phone_Available             0
Wealth_Rating               0
Income_Bracket              0
Event_Attendance            0
Email_Open_Rate             0
Location                    0
Consecutive_Giving_Years    0
Donation_Last_Year_2024     0
Total_Lifetime_Giving       0
Is_Donor_2025               0
dtype: int64

Total Missing Values:
0


In [13]:
# COMMAND: Check duplicate records

duplicate_count = df.duplicated().sum()

print("Duplicate Records:", duplicate_count)

Duplicate Records: 0


In [14]:
# COMMAND: Remove duplicate records if they exist

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates :", after)
print("Duplicates removed             :", before - after)

Rows before removing duplicates: 10000
Rows after removing duplicates : 10000
Duplicates removed             : 0


In [15]:
# COMMAND: Analyze target variable Is_Donor_2025

print("Target Variable:")
print("Is_Donor_2025")

print("\nTarget Distribution:")
print("=" * 60)

print(df["Is_Donor_2025"].value_counts())

print("\nTarget Percentage:")
print("=" * 60)

target_percentage = (
    df["Is_Donor_2025"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(target_percentage)

Target Variable:
Is_Donor_2025

Target Distribution:
Is_Donor_2025
Yes    5740
No     4260
Name: count, dtype: int64

Target Percentage:
Is_Donor_2025
Yes    57.4
No     42.6
Name: proportion, dtype: float64


In [16]:
# COMMAND: Validate target variable values

valid_targets = {"Yes", "No"}

actual_targets = set(
    df["Is_Donor_2025"].dropna().unique()
)

print("Actual target values:")
print(actual_targets)

if actual_targets.issubset(valid_targets):

    print("\nTarget validation successful!")

else:

    print("\nWARNING: Unexpected target values found!")

Actual target values:
{'No', 'Yes'}

Target validation successful!


In [17]:
# COMMAND: Separate input features X and target y

import pandas as pd

# Remove Alumni_ID because it is only an identifier.
# Remove Is_Donor_2025 because it is the target we want to predict.

X = df.drop(
    columns=[
        "Alumni_ID",
        "Is_Donor_2025"
    ]
)

y = df["Is_Donor_2025"].map({
    "No": 0,
    "Yes": 1
})

print("Features X shape:")
print(X.shape)

print("\nTarget y shape:")
print(y.shape)

print("\nTarget values:")
print(y.value_counts())

Features X shape:
(10000, 16)

Target y shape:
(10000,)

Target values:
Is_Donor_2025
1    5740
0    4260
Name: count, dtype: int64


In [18]:
# COMMAND: Display all features used for machine learning

print("Features used for training:")
print("=" * 60)

for number, column in enumerate(
    X.columns,
    start=1
):

    print(
        f"{number}. {column}"
    )

print("\nTotal Features:", len(X.columns))

Features used for training:
1. Age
2. Gender
3. Graduation_Year
4. Degree_Level
5. Major
6. Alumni_Status
7. Email_Available
8. Phone_Available
9. Wealth_Rating
10. Income_Bracket
11. Event_Attendance
12. Email_Open_Rate
13. Location
14. Consecutive_Giving_Years
15. Donation_Last_Year_2024
16. Total_Lifetime_Giving

Total Features: 16


In [19]:
# COMMAND: Identify numerical features

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Numerical Features:")
print("=" * 60)

for feature in numerical_features:
    print(feature)

print("\nTotal Numerical Features:")
print(len(numerical_features))

Numerical Features:
Age
Graduation_Year
Event_Attendance
Email_Open_Rate
Consecutive_Giving_Years
Donation_Last_Year_2024
Total_Lifetime_Giving

Total Numerical Features:
7


In [20]:
# COMMAND: Identify categorical features

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical Features:")
print("=" * 60)

for feature in categorical_features:
    print(feature)

print("\nTotal Categorical Features:")
print(len(categorical_features))

Categorical Features:
Gender
Degree_Level
Major
Alumni_Status
Email_Available
Phone_Available
Wealth_Rating
Income_Bracket
Location

Total Categorical Features:
9


In [21]:
# COMMAND: Split dataset into training and testing data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("TRAINING DATA")
print("=" * 60)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTESTING DATA")
print("=" * 60)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

TRAINING DATA
X_train: (8000, 16)
y_train: (8000,)

TESTING DATA
X_test: (2000, 16)
y_test: (2000,)


In [22]:
# COMMAND: Create numerical preprocessing pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)

print("Numerical preprocessing created successfully!")

Numerical preprocessing created successfully!


In [23]:
# COMMAND: Create categorical preprocessing pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

print("Categorical preprocessing created successfully!")

Categorical preprocessing created successfully!


In [24]:
# COMMAND: Combine numerical and categorical preprocessing

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numeric_transformer,
            numerical_features
        ),

        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

print("Combined preprocessor created successfully!")

Combined preprocessor created successfully!


In [25]:
# COMMAND: Create Random Forest classification model

from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

print("Random Forest model created successfully!")

print("\nNumber of trees:", 300)
print("Random state:", 42)
print("Class weight: balanced")

Random Forest model created successfully!

Number of trees: 300
Random state: 42
Class weight: balanced


In [26]:
# COMMAND: Combine preprocessing and Random Forest into one pipeline

from sklearn.pipeline import Pipeline

model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            random_forest_model
        )
    ]
)

print("Complete ML pipeline created successfully!")

print("\nPipeline:")
print(model_pipeline)

Complete ML pipeline created successfully!

Pipeline:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Age', 'Graduation_Year',
                                                   'Event_Attendance',
                                                   'Email_Open_Rate',
                                                   'Consecutive_Giving_Years',
                                                   'Donation_Last_Year_2024',
                                                   'Total_Lifetime_Giving']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   Simpl

In [27]:
# COMMAND: Train Random Forest model

print("=" * 60)
print("MODEL TRAINING STARTED")
print("=" * 60)

model_pipeline.fit(
    X_train,
    y_train
)

print("\nMODEL TRAINING COMPLETED SUCCESSFULLY!")

MODEL TRAINING STARTED

MODEL TRAINING COMPLETED SUCCESSFULLY!


In [28]:
# COMMAND: Generate predictions on test data

y_pred = model_pipeline.predict(
    X_test
)

print("Predictions generated successfully!")

print("\nNumber of predictions:")
print(len(y_pred))

print("\nFirst 10 predictions:")
print(y_pred[:10])

Predictions generated successfully!

Number of predictions:
2000

First 10 predictions:
[1 0 0 0 0 0 0 1 0 0]


In [29]:
# COMMAND: Generate probability of donation

y_probability = model_pipeline.predict_proba(
    X_test
)[:, 1]

print("Donation probabilities generated successfully!")

print("\nFirst 10 probability values:")

for probability in y_probability[:10]:

    print(
        f"{probability:.4f}"
    )

Donation probabilities generated successfully!

First 10 probability values:
0.8300
0.2033
0.2567
0.2933
0.3167
0.2200
0.1233
0.9300
0.1800
0.4333


In [30]:
# COMMAND: Calculate model accuracy

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Accuracy: {accuracy:.4f}"
)

Accuracy: 0.8000


In [31]:
# COMMAND: Calculate model precision

from sklearn.metrics import precision_score

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

print(
    f"Precision: {precision:.4f}"
)

Precision: 0.8979


In [32]:
# COMMAND: Calculate model recall

from sklearn.metrics import recall_score

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

print(
    f"Recall: {recall:.4f}"
)

Recall: 0.7352


In [33]:
# COMMAND: Calculate F1 score

from sklearn.metrics import f1_score

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print(
    f"F1 Score: {f1:.4f}"
)

F1 Score: 0.8084


In [34]:
# COMMAND: Calculate ROC-AUC score

from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

print(
    f"ROC-AUC: {roc_auc:.4f}"
)

ROC-AUC: 0.8331


In [35]:
# COMMAND: Display complete model performance

print("=" * 60)
print("             MODEL PERFORMANCE")
print("=" * 60)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc:.4f}"
)

print("=" * 60)

             MODEL PERFORMANCE
Accuracy : 0.8000
Precision: 0.8979
Recall   : 0.7352
F1 Score : 0.8084
ROC-AUC  : 0.8331


In [36]:
# COMMAND: Generate detailed classification report

from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=[
        "No Donor",
        "Donor"
    ],
    zero_division=0
)

print(report)

              precision    recall  f1-score   support

    No Donor       0.71      0.89      0.79       852
       Donor       0.90      0.74      0.81      1148

    accuracy                           0.80      2000
   macro avg       0.81      0.81      0.80      2000
weighted avg       0.82      0.80      0.80      2000



In [37]:
# COMMAND: Generate confusion matrix

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print("=" * 40)

print(cm)

print("\nMatrix format:")
print(
    "[[True Negative   False Positive]"
)
print(
    " [False Negative  True Positive]]"
)

Confusion Matrix:
[[756  96]
 [304 844]]

Matrix format:
[[True Negative   False Positive]
 [False Negative  True Positive]]


In [38]:
# COMMAND: Ensure models directory exists

import os

models_directory = os.path.join(
    project_path,
    "models"
)

os.makedirs(
    models_directory,
    exist_ok=True
)

print("Models directory ready:")
print(models_directory)

Models directory ready:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models


In [39]:
# COMMAND: Save trained ML pipeline as PKL file

import joblib

joblib.dump(
    model_pipeline,
    model_path
)

print("=" * 60)
print("MODEL SAVED SUCCESSFULLY!")
print("=" * 60)

print("\nSaved model:")
print(model_path)

MODEL SAVED SUCCESSFULLY!

Saved model:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl


In [40]:
# COMMAND: Verify saved model file

import os

if os.path.exists(model_path):

    file_size = os.path.getsize(
        model_path
    )

    print("SUCCESS!")
    print("Model file exists.")
    print("Model path:")
    print(model_path)

    print("\nModel file size:")
    print(file_size, "bytes")

else:

    print("ERROR!")
    print("Model file was not found.")

SUCCESS!
Model file exists.
Model path:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl

Model file size:
98070971 bytes


In [41]:
# COMMAND: Test whether the saved model can be loaded

import joblib

loaded_model = joblib.load(
    model_path
)

print("Saved model loaded successfully!")
print("\nModel type:")
print(type(loaded_model))

Saved model loaded successfully!

Model type:
<class 'sklearn.pipeline.Pipeline'>


In [42]:
# COMMAND: Test prediction using the saved model

test_sample = X_test.head(5)

loaded_predictions = loaded_model.predict(
    test_sample
)

loaded_probabilities = loaded_model.predict_proba(
    test_sample
)[:, 1]

print("Prediction test successful!")

print("\nPredictions:")
print(loaded_predictions)

print("\nProbabilities:")
print(loaded_probabilities)

Prediction test successful!

Predictions:
[1 0 0 0 0]

Probabilities:
[0.83       0.20333333 0.25666667 0.29333333 0.31666667]


In [43]:
# COMMAND: Save model evaluation metrics to CSV

import pandas as pd
import os

results = {
    "Model": "Random Forest Classifier",
    "Dataset_Size": len(df),
    "Number_of_Features": X.shape[1],
    "Training_Size": len(X_train),
    "Testing_Size": len(X_test),
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1_Score": f1,
    "ROC_AUC": roc_auc
}

results_df = pd.DataFrame(
    [results]
)

results_directory = os.path.join(
    project_path,
    "results"
)

os.makedirs(
    results_directory,
    exist_ok=True
)

results_df.to_csv(
    results_path,
    index=False
)

print("Training results saved successfully!")

print("\nResults file:")
print(results_path)

Training results saved successfully!

Results file:
/content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/results/model_training_results.csv


In [44]:
# COMMAND: Display final model performance table

display(
    results_df
)

,Model,Dataset_Size,Number_of_Features,Training_Size,Testing_Size,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,Random Forest Classifier,10000,16,8000,2000,0.8,0.897872,0.735192,0.808429,0.833149


In [45]:
# COMMAND: Verify complete training phase

import os

print("=" * 70)
print("       ALUMNI DONOR PROPENSITY FORECASTER")
print("              TRAINING COMPLETE")
print("=" * 70)

print("\nDataset:")
print(
    "✓",
    os.path.exists(dataset_path),
    dataset_path
)

print("\nTrained Model:")
print(
    "✓",
    os.path.exists(model_path),
    model_path
)

print("\nTraining Results:")
print(
    "✓",
    os.path.exists(results_path),
    results_path
)

print("\nModel Metrics:")
print(
    f"✓ Accuracy : {accuracy:.4f}"
)

print(
    f"✓ Precision: {precision:.4f}"
)

print(
    f"✓ Recall   : {recall:.4f}"
)

print(
    f"✓ F1 Score : {f1:.4f}"
)

print(
    f"✓ ROC-AUC  : {roc_auc:.4f}"
)

print("\n" + "=" * 70)
print("MODEL TRAINING PHASE COMPLETED!")
print("=" * 70)

       ALUMNI DONOR PROPENSITY FORECASTER
              TRAINING COMPLETE

Dataset:
✓ True /content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/data/alumni_donor_dataset_2025.csv

Trained Model:
✓ True /content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/models/alumni_donor_model_pipeline.pkl

Training Results:
✓ True /content/drive/MyDrive/Alumni_Donor_Propensity_Forecaster/results/model_training_results.csv

Model Metrics:
✓ Accuracy : 0.8000
✓ Precision: 0.8979
✓ Recall   : 0.7352
✓ F1 Score : 0.8084
✓ ROC-AUC  : 0.8331

MODEL TRAINING PHASE COMPLETED!
